# Suite CONF — Cross-implementation conformance

Every implementation builds the same corpus (`mbse/Expressions/Conformance/Corpus.py`, mirrored statement for
statement in TypeScript) and commits its snapshots under `conformance/<implementation>/`. This suite checks this
implementation's files are current, that its JSON is byte-identical to every other implementation's, and that it reads
every other implementation's JSON and YAML back to the same terms. Regenerate with
`uv run python -m mbse.Expressions.Conformance.write`. The corpus has a case for Patterns and for every dialect, each a
term written with its dialect, then its kind; each case validates in its own dialect. One case is data, not a term:
`domains`, ISO/IEC 11404's dialect, its domains, owned by name, each kind its dialect's then its own, an enum and a
packed one among them; its domains validate. One is invalid syntax, `invalid`: a Python3 term holding an `error`, a `missing` term and literals with their
spelling, one of them holding no value but its problem; its problems are reported.

In [ ]:
from pathlib import Path

import yaml

from mbse.Expressions.Conformance.Corpus import CASES, DIALECTS, build
from mbse.Expressions.Conformance.write import render
from mbse.Expressions.Framework.Patterns import Domains as D, Expressions as E
from mbse.Expressions.Framework import Symbolics as SY, Terms as F
from mbse.Schemas.Framework import JSON, Plain, YAML

ROOT = Path("../../conformance")
OWN = "python3"
OTHERS = sorted(p.name for p in ROOT.iterdir() if p.is_dir() and p.name != OWN)
assert OTHERS == ["typescript5"], OTHERS

corpus = build()
snapshots = {case: Plain.ToPlain(store)(schema, root) for case, (schema, root, store) in corpus.items()}

<a id="CONF-01"></a>
## CONF-01 · This implementation's committed files are current

In [ ]:
for name, text in render(corpus).items():
    assert (ROOT / OWN / name).read_text(encoding="utf-8") == text, f"{name} is stale: regenerate the corpus"

<a id="CONF-02"></a>
## CONF-02 · Every implementation has the same cases, and JSON is byte-identical across implementations

In [ ]:
for name in [OWN, *OTHERS]:
    assert sorted(p.name for p in (ROOT / name).iterdir()) == sorted(f"{c}.{e}" for c in CASES for e in ("json", "yaml")), name
for other in OTHERS:
    for case in CASES:
        mine = (ROOT / OWN / f"{case}.json").read_bytes()
        theirs = (ROOT / other / f"{case}.json").read_bytes()
        assert mine == theirs, f"{case}.json differs from {other}"

<a id="CONF-03"></a>
## CONF-03 · Every implementation's YAML reads back to exactly this implementation's snapshot, also under YAML 1.1

In [ ]:
for other in [OWN, *OTHERS]:
    for case in CASES:
        text = (ROOT / other / f"{case}.yaml").read_text(encoding="utf-8")
        assert YAML.loads(text) == snapshots[case], f"{other}/{case}.yaml"
        assert yaml.safe_load(text) == snapshots[case], f"{other}/{case}.yaml under YAML 1.1"

<a id="CONF-04"></a>
## CONF-04 · Every implementation's JSON and YAML deserialize into this implementation's store, and validate

In [ ]:
for other in [OWN, *OTHERS]:
    for case, (schema, root, store) in corpus.items():
        dialect = DIALECTS.get(case, E.DIALECT)
        for suffix, decode in [("json", JSON.FromJSON(store)), ("yaml", YAML.FromYAML(store))]:
            restored = decode(schema, (ROOT / other / f"{case}.{suffix}").read_bytes())
            assert Plain.ToPlain(store)(schema, restored) == snapshots[case], f"{other}/{case}.{suffix}"
            if case == "domains":  # ISO/IEC 11404's dialect, a singleton: read back into itself, its domains valid
                assert restored is root and all(D.read(entry.domain).validate() == [] for entry in restored.domains)
                names = ["boolean", "integer", "octetstring", "octet", "state", "mode"]
                assert [str(e.name) for e in restored.domains] == names and list(restored.operations) == []
                continue
            assert F.same(restored, root), f"{other}/{case}.{suffix}"
            problems = dialect.validate(restored, bound=SY.free(restored, dialect))
            if case == "invalid":  # invalid syntax and a literal's problem, kept and reported
                assert problems == ["left: left: right: invalid syntax '(x +* 1)'",
                                    "left: right: leading zeros in a decimal integer literal are not permitted",
                                    "right: missing expression"], (other, problems)
                continue
            assert problems == [], (other, case, suffix)